# Model Download And Swap Guide

Use this notebook when you want to add or replace models used by the app.

The app is intentionally self-contained inside `app/`. Keep downloaded models under `app/backend/...` so the project can run after deleting folders outside `app`.

## Current Model Locations

| Branch | Current Location | Code That Loads It |
| --- | --- | --- |
| Telemetry XGBoost | `app/backend/XGboost/telemetry_risk` | `app/backend/scenario_runtime.py` |
| DINOv2 Vision | `app/backend/vision_dinov2/facebook_dinov2_base` | `app/backend/vision_runtime.py` |
| DINOv2 Patch Memory | `app/backend/vision_dinov2/normal_patch_memory.pt` | `app/backend/vision_runtime.py` |
| Hugging Face RCA LLM | configurable with `HF_LLM_MODEL` | `app/backend/llm_rca_runtime.py` |
| RAG Knowledge | `app/backend/knowledge_rag` | `app/backend/rag_runtime.py` |

## Important

Do not delete these folders if you want the app to keep working:

```text
app/backend/XGboost
app/backend/vision_dinov2
app/backend/knowledge_rag
app/frontend
app/backend
```

In [ ]:
from pathlib import Path

APP = Path.cwd() if Path.cwd().name == "app" else Path.cwd() / "app"
BACKEND = APP / "backend"

paths = {
    "telemetry_xgboost": BACKEND / "XGboost" / "telemetry_risk",
    "vision_dinov2": BACKEND / "vision_dinov2",
    "hf_llm_models": BACKEND / "hf_models",
    "rag": BACKEND / "knowledge_rag",
}

for name, path in paths.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"{name}: {path.resolve()}")

## 1. Download A Hugging Face LLM For RCA Reasoning

The backend uses `HF_LLM_MODEL` in `app/backend/llm_rca_runtime.py`.

Recommended small default:

```text
Qwen/Qwen2.5-0.5B-Instruct
```

Better but heavier options:

```text
Qwen/Qwen2.5-1.5B-Instruct
microsoft/Phi-3.5-mini-instruct
mistralai/Mistral-7B-Instruct-v0.3
```

For this laptop/demo, use a small model unless you know GPU memory is enough.

After download, set:

```powershell
$env:HF_LLM_MODEL="app/backend/hf_models/Qwen2.5-0.5B-Instruct"
```

or permanently change `DEFAULT_HF_MODEL` in `app/backend/llm_rca_runtime.py`.

In [ ]:
# Optional: download a Hugging Face LLM into app/backend/hf_models.
# Run this only when you want to download model weights.

from huggingface_hub import snapshot_download

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
local_dir = BACKEND / "hf_models" / "Qwen2.5-0.5B-Instruct"

# Uncomment to download:
# snapshot_download(repo_id=model_id, local_dir=local_dir)
# print(f"Downloaded {model_id} to {local_dir.resolve()}")

## 2. Replace DINOv2 Vision Model

Current expected folder:

```text
app/backend/vision_dinov2/facebook_dinov2_base
```

`vision_runtime.py` loads it here:

```python
processor = AutoImageProcessor.from_pretrained(DINO_LOCAL_DIR)
model = AutoModel.from_pretrained(DINO_LOCAL_DIR)
```

If you download a different DINOv2 model, either:

1. Put it in the same folder path, replacing the current files, or
2. Change `DINO_LOCAL_DIR` in `app/backend/vision_runtime.py`.

Common model ids:

```text
facebook/dinov2-small
facebook/dinov2-base
facebook/dinov2-large
```

Important: if the DINO model changes, regenerate `normal_patch_memory.pt` with clean/normal reference images, otherwise anomaly scores may be wrong.

In [ ]:
# Optional: download a DINOv2 model locally.
# This only downloads the model. Rebuilding normal_patch_memory.pt is a separate training/calibration step.

from transformers import AutoImageProcessor, AutoModel

dinov2_model_id = "facebook/dinov2-base"
dinov2_dir = BACKEND / "vision_dinov2" / "facebook_dinov2_base"

# Uncomment to download/replace:
# processor = AutoImageProcessor.from_pretrained(dinov2_model_id)
# model = AutoModel.from_pretrained(dinov2_model_id)
# processor.save_pretrained(dinov2_dir)
# model.save_pretrained(dinov2_dir)
# print(f"Saved DINOv2 to {dinov2_dir.resolve()}")

## 3. Replace Telemetry XGBoost Models

Current expected folder:

```text
app/backend/XGboost/telemetry_risk
```

Required files:

```text
xgb_rul_model.joblib
xgb_risk_model.joblib
feature_columns.json
metadata.json
```

`scenario_runtime.py` builds runtime features using `feature_columns.json`. If you retrain models, make sure this file matches exactly the columns used during training.

If you move the folder, update:

```python
TELEMETRY_MODEL_DIR = Path(__file__).resolve().parent / "XGboost" / "telemetry_risk"
```

in `app/backend/scenario_runtime.py`.

In [ ]:
# Check telemetry model files exist.

telemetry_dir = BACKEND / "XGboost" / "telemetry_risk"
required = [
    "xgb_rul_model.joblib",
    "xgb_risk_model.joblib",
    "feature_columns.json",
    "metadata.json",
]

for name in required:
    path = telemetry_dir / name
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

## 4. Rebuild RAG Indexes After Adding Documents

Normal uploads through `/rag.html` already update these files:

```text
app/backend/knowledge_rag/uploaded_documents.jsonl
app/backend/knowledge_rag/text_vector_index.jsonl
app/backend/knowledge_rag/visual_index.jsonl
app/backend/knowledge_rag/work_order_index.jsonl
```

If you manually edit or copy RAG files, rebuild text vectors with this cell.

In [ ]:
import sys

sys.path.insert(0, str(BACKEND))
from rag_runtime import collection_summary, rebuild_text_vector_index

# Uncomment if you manually changed RAG JSONL files:
# rebuild_text_vector_index()

collection_summary()

## 5. Restart The Server After Model Changes

After changing models or environment variables, restart:

```powershell
python app/backend/server.py
```

Then open:

```text
http://127.0.0.1:8000
http://127.0.0.1:8000/rag.html
```

## Quick Health Checks

```text
GET /api/health
GET /api/rag/documents
POST /api/infer
POST /api/work-orders/search
```